In [ ]:
'''
encontrar al asesino a partir de iforme de la escena del crimen que ocurrió el 15 de enero de 2018
en SQL City
'''

In [1]:
# Importamos el módulo sqlite3, que permite a Python "hablar" con bases de datos SQLite[cite: 7].
import sqlite3
import pandas as pd
import numpy as np

In [2]:
# --- 1. CONEXIÓN A LA BASE DE DATOS ---
# Establecemos el "túnel" de comunicación con el archivo que contiene las tablas del crimen[cite: 7].
# Si el archivo no existe en la carpeta, Python creará uno vacío (lo cual daría error más adelante).
conn = sqlite3.connect('sql-murder-mystery.db')

# El 'cursor' es como un puntero o un mando a distancia; lo usamos para enviar órdenes SQL a través de la conexión.
cursor = conn.cursor()

# Definimos una función para no repetir código. 
# Recibe una consulta (query) y parámetros opcionales (params) para evitar ataques de inyección SQL.
def ejecutar_consulta(query, params=()):
    # El cursor ejecuta la sentencia SQL enviada.
    cursor.execute(query, params)
    # fetchall() recoge TODAS las filas que la base de datos devuelve como resultado y las mete en una lista.
    return cursor.fetchall()

In [5]:
# ---------------------------------------------------------
# PASO 1: Consultar el reporte de la escena del crimen
# Iniciamos la investigación con los únicos datos que recordamos[cite: 11].
# ---------------------------------------------------------
print("--- Reporte de la Escena del Crimen ---")

# Redactamos la instrucción SQL: "Selecciona la descripción de la tabla de reportes donde..."
# Filtramos por fecha exacta (20180115), tipo 'murder' (asesinato) y ciudad 'SQL City'[cite: 11, 50, 53, 55].
query_reporte = """
SELECT description 
FROM crime_scene_report 
WHERE date = 20180115 AND type = 'murder' AND city = 'SQL City'
"""

# Ejecutamos la función y guardamos el resultado.
reporte = ejecutar_consulta(query_reporte)

# accedemos a reporte[0][0]:
# el primer [0] es la primera fila devuelta.
# el segundo [0] es la primera columna de esa fila (la descripción).
print(reporte[0][0])



--- Reporte de la Escena del Crimen ---
Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".


In [6]:
# ---------------------------------------------------------
# PASO 2: Localizar a los testigos y leer sus declaraciones
# El reporte anterior nos dio pistas sobre dos vecinos que vieron algo[cite: 17].
# ---------------------------------------------------------
print("\n--- Declaraciones de los Testigos ---")

# Aquí usamos un JOIN para unir la tabla 'person' (datos del ciudadano) con 'interview' (lo que dijo)[cite: 17, 18, 21].
# La unión se hace donde el ID de la persona coincida en ambas tablas[cite: 22, 18].
query_testigos = """
SELECT p.name, i.transcript 
FROM person p
JOIN interview i ON p.id = i.person_id
WHERE 
    -- Condición 1: El testigo que vive en la última casa de 'Northwestern Dr'.
    -- Usamos una subconsulta (SELECT MAX...) para hallar el número de portal más alto automáticamente[cite: 24].
    (p.address_street_name = 'Northwestern Dr' 
       AND p.address_number = (SELECT MAX(address_number) FROM person WHERE address_street_name = 'Northwestern Dr'))
    OR 
    -- Condición 2: La testigo llamada Annabel que vive en 'Franklin Ave'[cite: 24].
    -- 'LIKE' con '%' busca nombres que empiecen por Annabel (por si tiene apellidos).
    (p.name LIKE 'Annabel%' AND p.address_street_name = 'Franklin Ave')
"""

testigos = ejecutar_consulta(query_testigos)

# 'testigos' es una lista de filas. Usamos un bucle 'for' para imprimir cada una.
for t in testigos:
    # t[0] es el nombre, t[1] es la transcripción de su entrevista.
    print(f"{t[0]}: {t[1]}")




--- Declaraciones de los Testigos ---
Morty Schapiro: I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".
Annabel Miller: I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


--Morty Saphiro: oye un disparo y un hombre echa a correr, es un miembro oro del gimnasio Get Fit Now, su número de socio comienza con '48Z' y se monta en un coche con matrícula que contiene 'H42W'.
--Annabel Miller: El asesino fue al gimnasio el 9 de enero.

Podemos mirar el checkin de quienes fueron al gym el 9 de enero. Podemos mirar a quién pertenece la matrícula

In [7]:
# ---------------------------------------------------------
# PASO 3: Identificar al asesino con las pistas obtenidas
# Ahora cruzamos las declaraciones de los testigos con los registros del gimnasio y tráfico[cite: 14, 27].
# ---------------------------------------------------------
print("\n--- Identificando al Asesino ---")

# Esta consulta es más compleja porque une 4 tablas: gimnasio, personas, licencias y registros de entrada[cite: 19, 21, 27, 14].
query_asesino = """
SELECT p.name
FROM get_fit_now_member gfm
JOIN person p ON gfm.person_id = p.id
JOIN drivers_license dl ON p.license_id = dl.id
JOIN get_fit_now_check_in ci ON gfm.id = ci.membership_id
WHERE 
    gfm.id LIKE '48Z%'              -- El ID de socio empieza por 48Z (pista del testigo)[cite: 15].
    AND gfm.membership_status = 'gold' -- El testigo dijo que era nivel 'Gold'[cite: 31].
    AND dl.plate_number LIKE '%H42W%' -- La matrícula contiene H42W.
    AND ci.check_in_date = 20180109    -- Annabel lo vio en el gimnasio este día[cite: 16].
"""

asesino = ejecutar_consulta(query_asesino)

# Guardamos el nombre para el siguiente paso.
nombre_asesino = asesino[0][0]
print(f"¡El asesino es: {nombre_asesino}!")




--- Identificando al Asesino ---
¡El asesino es: Jeremy Bowers!


In [ ]:
# ---------------------------------------------------------
# PASO 4: ¡Bonus! Encontrar al autor intelectual (quien lo contrató)
# Interrogamos al culpable para llegar a la "Mente Maestra"[cite: 17].
# ---------------------------------------------------------
print("\n--- Interrogatorio del Asesino ---")

# Buscamos la entrevista del asesino que acabamos de atrapar.
# El '?' es un marcador de posición que se sustituye por (nombre_asesino,) de forma segura.
query_autor_intelectual = """
SELECT i.transcript 
FROM interview i
JOIN person p ON i.person_id = p.id
WHERE p.name = ?
"""
confesion = ejecutar_consulta(query_autor_intelectual, (nombre_asesino,))
print(f"Confesión: {confesion[0][0]}")



In [8]:
# PASO FINAL: Filtrar por las características físicas de la confesión.
# Buscamos a una mujer (pelo rojo, altura específica) con un coche Tesla Model S[cite: 40, 42, 34, 45, 48].
query_cerebro = """
SELECT p.name
FROM person p
JOIN drivers_license dl ON p.license_id = dl.id
JOIN facebook_event_checkin fe ON p.id = fe.person_id
WHERE 
    dl.hair_color = 'red'             -- Pelo rojo[cite: 40].
    AND dl.car_make = 'Tesla'         -- Marca Tesla[cite: 45].
    AND dl.car_model = 'Model S'      -- Modelo Model S[cite: 48].
    -- La altura en el juego suele estar en pulgadas. 65-67 pulgadas son aprox 1.65m-1.70m.
    AND dl.height BETWEEN 65 AND 67   
    AND fe.event_name = 'SQL Symphony Concert' -- Asistió a este evento.
GROUP BY p.name
-- Filtramos por personas que asistieron 3 veces al evento (pista adicional del juego).
HAVING COUNT(*) = 3
"""

cerebro = ejecutar_consulta(query_cerebro)

# Imprimimos el nombre de la verdadera culpable que contrató al asesino[cite: 64].
print(f"La mente maestra detrás de todo es: {cerebro[0][0]}")



La mente maestra detrás de todo es: Miranda Priestly


In [ ]:
# Es vital cerrar la conexión para no dejar "hilos" abiertos en el sistema operativo.
conn.close()

In [10]:
# --- BLOQUE DE IMPORTACIÓN ---
# Importamos la librería estándar de Python para manejar bases de datos SQLite.
# No necesitas instalar nada extra, ya viene con Python[cite: 7].
import sqlite3

def resolver_misterio():
    """
    Esta función encapsula toda la lógica de la investigación.
    Usar una función ayuda a mantener el código organizado y profesional.
    """
    try:
        # --- CONEXIÓN Y CURSOR ---
        # 1. Establecemos la conexión física con el archivo de la base de datos.
        # Si el archivo no está en la misma carpeta, dará un error[cite: 7].
        conexion = sqlite3.connect('sql-murder-mystery.db')
        
        # 2. Creamos un 'cursor'. El cursor es como un puntero o un mando a distancia
        # que enviará nuestras órdenes (SQL) a la base de datos y traerá los datos de vuelta.
        cursor = conexion.cursor()
        
        print("--- PASO 1: Buscando el reporte del crimen ---")
        # El enunciado dice que el asesinato fue el 15 de enero de 2018 en SQL City.
        # En SQL, los números no llevan comillas, pero los textos (strings) sí.
        query_reporte = """
        SELECT description 
        FROM crime_scene_report 
        WHERE date = 20180115 
          AND type = 'murder' 
          AND city = 'SQL City';
        """
        # Enviamos la orden a la base de datos:
        cursor.execute(query_reporte)
        
        # Guardamos el resultado en la variable 'fila_reporte'. 
        # fetchone() se usa porque solo esperamos un único reporte para ese día y lugar.
        fila_reporte = cursor.fetchone()
        
        # Control de errores básico: si la consulta no devuelve nada, 'fila_reporte' será None.
        if fila_reporte is None:
            print("Error: No se encontró ningún reporte con esos datos.")
            return

        # Accedemos al índice [0] porque fetchone devuelve una lista/tupla, y la descripción es el primer elemento.
        descripcion_crimen = fila_reporte[0]
        print(f"Pista inicial: {descripcion_crimen}\n")

        print("--- PASO 2: Identificando y escuchando a los testigos ---")
        # Según el modelo de datos, las entrevistas están en la tabla 'interview'[cite: 17, 18].
        # Necesitamos unir ('JOIN') la tabla 'person' con 'interview' usando el ID de la persona[cite: 21, 22].
        # Buscamos a dos personas específicas basadas en la descripción del reporte:
        # - El que vive en el número más alto de 'Northwestern Dr'.
        # - Annabel, que vive en 'Franklin Ave'.
        query_testigos = """
        SELECT p.name, i.transcript 
        FROM person p
        JOIN interview i ON p.id = i.person_id
        WHERE (p.address_street_name = 'Northwestern Dr' 
               AND p.address_number = (SELECT MAX(address_number) FROM person WHERE address_street_name = 'Northwestern Dr'))
           OR (p.name LIKE 'Annabel%' AND p.address_street_name = 'Franklin Ave');
        """
        cursor.execute(query_testigos)
        
        # fetchall() devuelve una lista con todos los testigos encontrados (deberían ser 2).
        lista_testigos = cursor.fetchall()
        
        # Usamos un bucle 'for' para recorrer la lista y mostrar qué dijo cada uno.
        for nombre, testimonio in lista_testigos:
            print(f"Testigo: {nombre}")
            print(f"Dice: {testimonio}\n")

        print("--- PASO 3: Cruzando pistas para encontrar al asesino ---")
        # Aquí combinamos las pistas de los testimonios:
        # 1. Miembro del gimnasio 'Get Fit Now' (tabla get_fit_now_member)[cite: 19].
        # 2. El ID de socio empieza por '48Z' y es tipo 'Gold'[cite: 15, 31].
        # 3. Tiene un coche con matrícula que contiene 'H42W' (tabla drivers_license)[cite: 27, 44].
        
        query_asesino = """
        SELECT p.name 
        FROM person p
        -- Unimos con gimnasio para filtrar por socio
        JOIN get_fit_now_member m ON p.id = m.person_id
        -- Unimos con licencia para filtrar por coche/matrícula
        JOIN drivers_license dl ON p.license_id = dl.id
        WHERE m.id LIKE '48Z%' 
          AND m.membership_status = 'gold'
          AND dl.plate_number LIKE '%H42W%';
        """
        cursor.execute(query_asesino)
        resultado_final = cursor.fetchone()

        # Verificamos si encontramos a alguien que cumpla TODAS las condiciones.
        if resultado_final:
            # ¡ÉXITO! Imprimimos el nombre, que es lo que pide el entregable[cite: 64].
            print(f"¡ENCONTRADO! El nombre del verdadero asesino es: {resultado_final[0]}")
        else:
            print("No se encontró a nadie que coincida con todas las pistas.")

        # --- CIERRE DE RECURSOS ---
        # Es una buena práctica cerrar la conexión para que no consuma memoria.
        conexion.close()
        
    except sqlite3.Error as e:
        # Si algo falla (SQL mal escrito, tabla inexistente), Python saltará aquí.
        print(f"Ocurrió un error técnico con la base de datos: {e}")

# Estas dos líneas aseguran que la función se ejecute solo si abres este archivo directamente.
if __name__ == "__main__":
    resolver_misterio()

--- PASO 1: Buscando el reporte del crimen ---
Pista inicial: Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".

--- PASO 2: Identificando y escuchando a los testigos ---
Testigo: Morty Schapiro
Dice: I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".

Testigo: Annabel Miller
Dice: I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.

--- PASO 3: Cruzando pistas para encontrar al asesino ---
¡ENCONTRADO! El nombre del verdadero asesino es: Jeremy Bowers


-SELECT: Es para elegir qué columnas queremos ver (ej. el nombre).
-FROM: Indica en qué tabla está la información.
-JOIN: Es como un pegamento. Sirve para unir dos tablas que tienen algo en común (como el ID de una persona).
-WHERE: Es el filtro. Como en Excel, solo muestra las filas que cumplen la condición.
-LIKE y %: El símbolo % es un comodín. '48Z%' significa "cualquier cosa que empiece por 48Z".
-MAX(): Busca el valor más grande en una columna numérica, como el número de una calle.